# 구조화된 출력 — `output=`, `schema=`

JSON dict, Pydantic 모델, Enum 등으로 응답 형식을 제어.

In [ ]:
from llm_manager._internal import handler_client as llm_handler
from llm_manager._internal.handler_client import call

# llm_handler.configure(base_url="http://localhost:8100")


### JSON 출력 - dict schema

In [3]:
import json

response = call(
    "대한민국의 수도와 인구를 알려줘",
    schema={
        "type": "object",
        "properties": {
            "city": {"type": "string"},
            "population": {"type": "integer"},
        },
    },
)
data = json.loads(response.text)
print(data)

{'city': 'Seoul', 'population': 9400000}


### JSON 출력 - Pydantic 모델

In [4]:
from pydantic import BaseModel


class StockAnalysis(BaseModel):
    company: str
    sentiment: str
    confidence: float
    summary: str


response = call("삼성전자 실적 분석해줘", schema=StockAnalysis)
print(response.text)

data = json.loads(response.text)
print(data)

{"company":"삼성전자","sentiment":"Negative","confidence":0.85,"summary":"삼성전자의 2024년 3분기 잠정 영업이익은 약 9.1조 원으로 시장 기대치인 10조 원대를 하회하는 어닝 쇼크를 기록했습니다. HBM 사업의 상대적 지연과 범용 메모리 수요 둔화, 중국 업체의 추격 등이 주요 우려 사항으로 분석됩니다."}
{'company': '삼성전자', 'sentiment': 'Negative', 'confidence': 0.85, 'summary': '삼성전자의 2024년 3분기 잠정 영업이익은 약 9.1조 원으로 시장 기대치인 10조 원대를 하회하는 어닝 쇼크를 기록했습니다. HBM 사업의 상대적 지연과 범용 메모리 수요 둔화, 중국 업체의 추격 등이 주요 우려 사항으로 분석됩니다.'}


### JSON 출력 - 중첩 Pydantic + dict[str, int] (additionalProperties 검증)

`dict[str, int]` 필드는 Pydantic이 JSON Schema에서 `additionalProperties`를 생성한다.
Handler가 `response_json_schema`를 사용하므로 에러 없이 동작해야 한다.

In [5]:
from typing import Optional

from pydantic import BaseModel, Field


class CompanyInfo(BaseModel):
    name: str = Field(description="기업명")
    industry: str = Field(description="산업 분류")
    revenue: Optional[str] = Field(None, description="매출액")
    source_pages: Optional[dict[str, int]] = Field(
        None,
        description='필드별 출처 페이지 번호. 예: {"name": 1, "revenue": 3}',
    )


class Report(BaseModel):
    title: str
    companies: list[CompanyInfo]


# 스키마 확인 — additionalProperties 포함 여부
schema = Report.model_json_schema()
print(json.dumps(schema, indent=2, ensure_ascii=False))
print()

# LLM 호출 — Handler가 response_json_schema로 전달하므로 에러 없이 동작해야 함
response = call("삼성전자와 SK하이닉스에 대해 알려줘", schema=Report)
print(response.text)

{
  "$defs": {
    "CompanyInfo": {
      "properties": {
        "name": {
          "description": "기업명",
          "title": "Name",
          "type": "string"
        },
        "industry": {
          "description": "산업 분류",
          "title": "Industry",
          "type": "string"
        },
        "revenue": {
          "anyOf": [
            {
              "type": "string"
            },
            {
              "type": "null"
            }
          ],
          "default": null,
          "description": "매출액",
          "title": "Revenue"
        },
        "source_pages": {
          "anyOf": [
            {
              "additionalProperties": {
                "type": "integer"
              },
              "type": "object"
            },
            {
              "type": "null"
            }
          ],
          "default": null,
          "description": "필드별 출처 페이지 번호. 예: {\"name\": 1, \"revenue\": 3}",
          "title": "Source Pages"
        }
      },
      "

HandlerError: [provider] Gemini 클라이언트 에러: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Invalid JSON payload received. Unknown name "additional_properties" at \'generation_config.response_schema.properties[1].value.items.properties[3].value\': Cannot find field.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.BadRequest', 'fieldViolations': [{'field': 'generation_config.response_schema.properties[1].value.items.properties[3].value', 'description': 'Invalid JSON payload received. Unknown name "additional_properties" at \'generation_config.response_schema.properties[1].value.items.properties[3].value\': Cannot find field.'}]}]}} (400)

### Enum 분류

In [22]:
from enum import Enum


class Sentiment(Enum):
    POSITIVE = "Positive"
    NEGATIVE = "Negative"
    NEUTRAL = "Neutral"


response = call("키움증권의 2024년 실적은 매우 좋았다.", schema=Sentiment)
print(response.text)

Positive


### schema 없이 output만 사용

In [23]:
response = call(
    "한국 5대 도시의 이름과 인구를 JSON으로 알려줘",
    output="json",
)
print(response.text)

[
  {
    "rank": 1,
    "city_name": "서울특별시",
    "population": 9368367
  },
  {
    "rank": 2,
    "city_name": "부산광역시",
    "population": 3280815
  },
  {
    "rank": 3,
    "city_name": "인천광역시",
    "population": 3004386
  },
  {
    "rank": 4,
    "city_name": "대구광역시",
    "population": 2368670
  },
  {
    "rank": 5,
    "city_name": "대전광역시",
    "population": 1441272
  }
]
